In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import gymnasium as gym
import torch

import fishyrl as frl

ModuleNotFoundError: No module named 'gymnasium'

In [ ]:
# Create environment
env = gym.make("CartPole-v1")  # , render_mode="human"

# Reset environment
obs, info = env.reset(seed=42)

c:\Users\thema\.conda\envs\fishyrl\Lib\site-packages\pygame\pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


In [ ]:
# Initialize model
ACTION_DIM = 1
OBS_DIM = obs.shape[-1]

EMBEDDED_OBS_DIM = 4

STOCHASTIC_DIM = 6
DETERMINISTIC_DIM = 7
CATEGORICAL_BINS = 11
REWARD_BINS = 13

# TODO: TwoHot over rewards with reward and continue models
# Encoder-Decoder Models
encoder_model = frl.models.MLPEncoder(OBS_DIM, EMBEDDED_OBS_DIM)
decoder_model = frl.models.MLPDecoder(STOCHASTIC_DIM * CATEGORICAL_BINS + DETERMINISTIC_DIM, OBS_DIM)

# RSSM Models
recurrent_model = frl.models.RecurrentModel(STOCHASTIC_DIM * CATEGORICAL_BINS + ACTION_DIM, DETERMINISTIC_DIM)
representation_model = frl.models.MLP(EMBEDDED_OBS_DIM + DETERMINISTIC_DIM, [STOCHASTIC_DIM * CATEGORICAL_BINS])
transition_model = frl.models.MLP(DETERMINISTIC_DIM, [STOCHASTIC_DIM * CATEGORICAL_BINS])
rssm_model = frl.models.RSSM(recurrent_model, representation_model, transition_model, CATEGORICAL_BINS)

# Reward and Continue Models
reward_model = frl.models.MLP(STOCHASTIC_DIM * CATEGORICAL_BINS + DETERMINISTIC_DIM, REWARD_BINS, [512])  # Double-check num layers
continue_model = frl.models.MLP(STOCHASTIC_DIM * CATEGORICAL_BINS + DETERMINISTIC_DIM, 1, [512])  # Double-check num layers


In [ ]:
assert False

In [ ]:
# # Using initial state, which we would use to generate an action
# hidden_state = rssm_model.initial_hidden_state.unsqueeze(0)
# posterior = rssm_model.infer_stochastic(hidden_state, torch.randn(1, EMBEDDED_OBS_DIM))[1]

# # Generate and perform an action (using hidden state and posterior), then step the environment
# pass

# # Get new posterior
# for _ in range(5):
#     ret = rssm_model.forward(
#         torch.randn(1, ACTION_DIM),
#         hidden_state,
#         posterior,
#         torch.randn(1, EMBEDDED_OBS_DIM),
#     )
#     hidden_state = ret['hidden_state']
#     posterior = ret['posterior']

# # Imagine new prior
# prior = posterior
# for _ in range(5):
#     ret = rssm_model.forward(
#         torch.randn(1, ACTION_DIM),
#         hidden_state,
#         prior
#     )
#     hidden_state = ret['hidden_state']
#     prior = ret['prior']

In [ ]:
# Reset environment
obs, info = env.reset(seed=42)
obs = torch.tensor(obs).unsqueeze(0)

# Get initial hidden state and posterior
hidden_state = rssm_model.initial_hidden_state.unsqueeze(0)
posterior = rssm_model.infer_stochastic(hidden_state, obs)[1]

# Loop until episode is done
# TODO: Add buffer
# TODO: Make run for multiple episodes
terminated = False
while not terminated:
    # Generate action from hidden state and posterior, then step the environment
    action = torch.tensor(env.action_space.sample()).reshape((-1, ACTION_DIM))  # TODO
    obs, reward, terminated, truncated, info = env.step(action.cpu().item())
    obs = torch.tensor(obs).unsqueeze(0)

    # If terminated, break loop
    if terminated:
        break

    # Embed observation
    embedded_obs = encoder_model(obs)
    reconstructed_obs = decoder_model(torch.cat((posterior, hidden_state), dim=-1))

    # Get new posterior
    ret = rssm_model.forward(
        action,
        posterior,
        hidden_state,
        embedded_obs,
    )
    hidden_state = ret['hidden_state']
    posterior = ret['posterior']

In [ ]:
env.close()